In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
print(os.path.exists('/content/drive'))
print(os.listdir('/content/drive'))

True
['MyDrive']


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main 2.zip"
extract_path = "/content/drive/MyDrive/GP Files/HIOD"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

Dataset extracted to: /content/drive/MyDrive/GP Files/HIOD


In [ ]:
!pip install -q pyyaml tqdm scikit-learn opencv-python matplotlib

In [ ]:
import os

root = "/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main"

for current_root, dirs, files in os.walk(root):
    print(current_root)
    if len(files) > 0:
        print("  sample files:", files[:5])
    if len(dirs) > 0:
        print("  subdirs:", dirs[:10])
    print("-" * 80)
    # stop after a reasonable amount
    if current_root.count(os.sep) - root.count(os.sep) > 2:
        continue

/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main
  sample files: ['README.md']
  subdirs: ['image', 'labels', 'readme']
--------------------------------------------------------------------------------
/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main/image
  sample files: ['1']
  subdirs: ['image1', 'image2', 'image3', 'image4', 'image5']
--------------------------------------------------------------------------------
/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main/image/image1
  sample files: ['1', 'image_1.jpg', 'image_10.jpg', 'image_100.jpg', 'image_1000.jpg']
--------------------------------------------------------------------------------
/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main/image/image2
  sample files: ['2', 'image_1001.jpg', 'image_1002.jpg', 'image_1003.jpg', 'image_1004.jpg']
--------------------------------------------------------------------------------
/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-ma

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main"
PROJECT_DIR = "/content/drive/MyDrive/GP Files/HIOD"
CONFIG_PATH = os.path.join(PROJECT_DIR, "Code/config.yaml")

os.makedirs(PROJECT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("CONFIG_PATH:", CONFIG_PATH)

BASE_DIR: /content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main
PROJECT_DIR: /content/drive/MyDrive/GP Files/HIOD
CONFIG_PATH: /content/drive/MyDrive/GP Files/HIOD/Code/config.yaml


In [ ]:
import yaml

config = {
    "project": {
        "seed": 42,
        "device": "cuda",
        "num_workers": 2,
        "out_dir": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_2",
    },

    "data": {
        "base_dir": "/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main",
        "image_label_pairs": [
            {"images": "image/image1", "labels": "labels/label1"},
            {"images": "image/image2", "labels": "labels/label2"},
            {"images": "image/image3", "labels": "labels/label3"},
            {"images": "image/image4", "labels": "labels/label4"},
            {"images": "image/image5", "labels": "labels/label5"},
        ],
        "val_ratio": 0.2,
        "class_map": {
            "staff": "medical_staff",
            "patient": "patient",
            "visitor": "other",
            "person": "other",
        },
    },

    "preprocess": {
        "output_root": "/content/drive/MyDrive/GP Files/HIOD/hiod_role_crops",
        "crop_padding": 0.15,
        "min_box_area": 2500,
        "max_aspect_ratio": 6.0,
        "save_debug_samples": 50,
        "enable_dedup": True,
        "dedup_phash_thresh": 6,
    },

    "train": {
        "model": "mobilenet_v3_large",
        "pretrained": True,
        "num_classes": 3,
        "img_size": 224,
        "batch_size": 32,
        "epochs": 45,
        "lr": 0.0003,
        "min_lr": 0.00001,
        "weight_decay": 0.0001,
        "label_smoothing": 0.03,
        "amp": True,
        "use_class_weights": True,
        "use_weighted_sampler": True,
        "early_stopping_patience": 8,
    },

    "augment": {
        "hflip": 0.5,
        "color_jitter": 0.25,
        "random_erasing": 0.15,
        "random_crop_scale": [0.8, 1.0],
        "rotation_deg": 10,
        "translate": 0.05,
    },

    "eval": {
        "save_confusion_matrix": True,
        "confusion_matrix_path": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_2/confusion_matrix.png",
        "normalized_confusion_matrix_path": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_2/confusion_matrix_normalized.png",
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Wrote config to:", CONFIG_PATH)

Wrote config to: /content/drive/MyDrive/GP Files/HIOD/Code/config.yaml


In [ ]:
import os
import cv2
import yaml
import json
import random
import shutil
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

ROLE_ORDER = ["patient", "medical_staff", "other"]


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def expand_bbox(x1, y1, x2, y2, pad_frac, W, H):
    w = x2 - x1
    h = y2 - y1
    pad_w = w * pad_frac
    pad_h = h * pad_frac
    nx1 = clamp(x1 - pad_w, 0, W - 1)
    ny1 = clamp(y1 - pad_h, 0, H - 1)
    nx2 = clamp(x2 + pad_w, 0, W - 1)
    ny2 = clamp(y2 + pad_h, 0, H - 1)
    return nx1, ny1, nx2, ny2


def phash(img_bgr, hash_size=8):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(np.float32(gray))
    dct_low = dct[:hash_size, :hash_size]
    med = np.median(dct_low)
    return (dct_low > med).astype(np.uint8).flatten()


def hamming(a, b):
    return int(np.count_nonzero(a != b))


def parse_voc_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objs = []
    for obj in root.findall("object"):
        name = obj.findtext("name")
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        x1 = float(bnd.findtext("xmin"))
        y1 = float(bnd.findtext("ymin"))
        x2 = float(bnd.findtext("xmax"))
        y2 = float(bnd.findtext("ymax"))
        objs.append((name, x1, y1, x2, y2))
    return objs


def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def collect_image_xml_pairs(cfg):
    """
    Reads the exact HIOD structure:
      base_dir/
        image/image1 ... image5
        labels/label1 ... label5
    and returns list of (img_path, xml_path, group_name)
    """
    base_dir = cfg["data"]["base_dir"]
    pairs = cfg["data"]["image_label_pairs"]

    items = []
    for pair in pairs:
        img_rel = pair["images"]
        lbl_rel = pair["labels"]

        img_dir = os.path.join(base_dir, img_rel)
        lbl_dir = os.path.join(base_dir, lbl_rel)
        group_name = os.path.basename(img_rel)   # image1, image2, ...

        img_paths = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            img_paths.extend(list(Path(img_dir).glob(ext)))

        for img_path in sorted(img_paths):
            stem = Path(img_path).stem
            xml_path = os.path.join(lbl_dir, stem + ".xml")
            if os.path.exists(xml_path):
                items.append((str(img_path), xml_path, group_name))

    return items


def preprocess_hiod(config_path):
    cfg = load_config(config_path)
    set_seed(cfg["project"]["seed"])

    class_map = cfg["data"]["class_map"]
    out_root = cfg["preprocess"]["output_root"]
    pad = float(cfg["preprocess"]["crop_padding"])
    min_area = float(cfg["preprocess"]["min_box_area"])
    max_ar = float(cfg["preprocess"]["max_aspect_ratio"])
    save_debug = int(cfg["preprocess"]["save_debug_samples"])
    enable_dedup = bool(cfg["preprocess"]["enable_dedup"])
    phash_thresh = int(cfg["preprocess"]["dedup_phash_thresh"])
    val_ratio = float(cfg["data"]["val_ratio"])

    items = collect_image_xml_pairs(cfg)
    print(f"Found {len(items)} image-xml pairs")

    # split by image, shuffled
    train_items, val_items = train_test_split(
        items,
        test_size=val_ratio,
        random_state=cfg["project"]["seed"],
        shuffle=True
    )

    split_items = {
        "train": train_items,
        "val": val_items
    }

    # Remove old output
    if os.path.exists(out_root):
        print(f"[INFO] Removing existing output: {out_root}")
        shutil.rmtree(out_root)

    # Create only valid class folders under train/val
    for split in ["train", "val"]:
        for role in ROLE_ORDER:
            ensure_dir(os.path.join(out_root, split, role))

    # Debug folders OUTSIDE train/val
    debug_train_dir = os.path.join(out_root, "_debug_train")
    debug_val_dir = os.path.join(out_root, "_debug_val")
    ensure_dir(debug_train_dir)
    ensure_dir(debug_val_dir)

    stats = {sp: {r: 0 for r in ROLE_ORDER} for sp in ["train", "val"]}
    skipped = {"train": 0, "val": 0}
    seen_hashes = {}

    def dedup_key(split, role):
        return f"{split}:{role}"

    for split in ["train", "val"]:
        pbar = tqdm(split_items[split], desc=f"Preprocess [{split}]")

        for img_path, xml_path, group_name in pbar:
            img = cv2.imread(img_path)
            if img is None:
                skipped[split] += 1
                continue

            H, W = img.shape[:2]
            stem = Path(img_path).stem

            try:
                objs = parse_voc_xml(xml_path)
            except Exception:
                skipped[split] += 1
                continue

            crop_idx = 0
            for label, x1, y1, x2, y2 in objs:
                if label not in class_map:
                    continue

                role = class_map[label]
                if role not in ROLE_ORDER:
                    continue

                bw = max(1.0, x2 - x1)
                bh = max(1.0, y2 - y1)
                area = bw * bh
                ar = max(bw / bh, bh / bw)

                if area < min_area or ar > max_ar:
                    continue

                ex1, ey1, ex2, ey2 = expand_bbox(x1, y1, x2, y2, pad, W, H)
                ex1, ey1, ex2, ey2 = map(int, [ex1, ey1, ex2, ey2])

                if ex2 <= ex1 or ey2 <= ey1:
                    continue

                crop = img[ey1:ey2, ex1:ex2].copy()
                if crop.size == 0:
                    continue

                # dedup per split + role
                if enable_dedup:
                    key = dedup_key(split, role)
                    hsh = phash(crop)
                    prev_list = seen_hashes.get(key, [])
                    is_dup = any(hamming(hsh, p) <= phash_thresh for p in prev_list[-500:])
                    if is_dup:
                        continue
                    prev_list.append(hsh)
                    seen_hashes[key] = prev_list

                out_name = f"{group_name}__{stem}__{label}__{crop_idx:03d}.jpg"
                out_path = os.path.join(out_root, split, role, out_name)
                cv2.imwrite(out_path, crop)
                stats[split][role] += 1

                # save debug examples outside class folders
                if stats[split][role] <= save_debug:
                    dbg = crop.copy()
                    cv2.putText(
                        dbg,
                        f"{label}->{role}",
                        (5, 20),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2
                    )
                    debug_dir = debug_train_dir if split == "train" else debug_val_dir
                    cv2.imwrite(os.path.join(debug_dir, out_name), dbg)

                crop_idx += 1

    meta_path = os.path.join(out_root, "preprocess_stats.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump({
            "stats": stats,
            "skipped_images": skipped,
            "role_order": ROLE_ORDER,
            "train_image_count": len(train_items),
            "val_image_count": len(val_items)
        }, f, indent=2)

    print("\n=== Done ===")
    print(json.dumps(stats, indent=2))
    print("Skipped images:", skipped)
    print("Wrote:", meta_path)
    print("ImageFolder root:", out_root)
    print("Debug folders:", debug_train_dir, debug_val_dir)

In [ ]:
preprocess_hiod(CONFIG_PATH)

Found 4347 image-xml pairs
[INFO] Removing existing output: /content/drive/MyDrive/GP Files/HIOD/hiod_role_crops


Preprocess [val]: 100%|██████████| 870/870 [12:24<00:00,  1.17it/s]


=== Done ===
{
  "train": {
    "patient": 425,
    "medical_staff": 1326,
    "other": 391
  },
  "val": {
    "patient": 103,
    "medical_staff": 340,
    "other": 97
  }
}
Skipped images: {'train': 0, 'val': 0}
Wrote: /content/drive/MyDrive/GP Files/HIOD/hiod_role_crops/preprocess_stats.json
ImageFolder root: /content/drive/MyDrive/GP Files/HIOD/hiod_role_crops
Debug folders: /content/drive/MyDrive/GP Files/HIOD/hiod_role_crops/_debug_train /content/drive/MyDrive/GP Files/HIOD/hiod_role_crops/_debug_val


In [ ]:
import os
from collections import defaultdict

crop_root = "/content/drive/MyDrive/GP Files/HIOD/hiod_role_crops"
counts = defaultdict(dict)

for split in ["train", "val"]:
    for cls in ["patient", "medical_staff", "other"]:
        p = os.path.join(crop_root, split, cls)
        counts[split][cls] = len(os.listdir(p)) if os.path.isdir(p) else 0

counts

defaultdict(dict,
            {'train': {'patient': 425, 'medical_staff': 1326, 'other': 391},
             'val': {'patient': 103, 'medical_staff': 340, 'other': 97}})

In [ ]:
import shutil

SRC = "/content/drive/MyDrive/GP Files/HIOD/hiod_role_crops"
DST = "/content/hiod_role_crops"

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)
print("Copied dataset to:", DST)

Copied dataset to: /content/hiod_role_crops


In [ ]:
import os
import yaml
import json
import time
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt

ROLE_ORDER = ["medical_staff", "other", "patient"]


def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_transforms(cfg):
    img_size = int(cfg["train"]["img_size"])
    aug = cfg["augment"]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(
            img_size,
            scale=tuple(aug["random_crop_scale"])
        ),
        transforms.RandomHorizontalFlip(p=float(aug["hflip"])),
        transforms.RandomRotation(degrees=float(aug["rotation_deg"])),
        transforms.RandomAffine(
            degrees=0,
            translate=(float(aug["translate"]), float(aug["translate"]))
        ),
        transforms.ColorJitter(
            brightness=float(aug["color_jitter"]),
            contrast=float(aug["color_jitter"]),
            saturation=float(aug["color_jitter"]),
            hue=min(0.1, float(aug["color_jitter"]) / 2.0),
        ),
        transforms.ToTensor(),
        transforms.RandomErasing(
            p=float(aug["random_erasing"]),
            scale=(0.02, 0.12),
            ratio=(0.3, 3.3)
        ),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    return train_tf, val_tf


def make_model(cfg):
    name = cfg["train"]["model"]
    pretrained = bool(cfg["train"]["pretrained"])
    num_classes = int(cfg["train"]["num_classes"])

    if name == "mobilenet_v3_large":
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.mobilenet_v3_large(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Unknown model: {name}")


def compute_class_weights_from_dataset(dataset, class_order, device):
    """
    dataset.targets are integer class ids from ImageFolder.
    Returns tensor of weights aligned with dataset.class_to_idx order.
    """
    targets = np.array(dataset.targets)
    num_classes = len(dataset.classes)
    counts = np.bincount(targets, minlength=num_classes).astype(np.float32)

    # inverse frequency
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()

    print("Class counts:", {dataset.classes[i]: int(counts[i]) for i in range(num_classes)})
    print("Class weights:", {dataset.classes[i]: float(weights[i]) for i in range(num_classes)})

    return torch.tensor(weights, dtype=torch.float32, device=device), counts


def make_weighted_sampler(dataset):
    targets = np.array(dataset.targets)
    class_sample_count = np.bincount(targets, minlength=len(dataset.classes)).astype(np.float32)
    class_weights = 1.0 / np.maximum(class_sample_count, 1.0)
    sample_weights = class_weights[targets]
    sample_weights = torch.from_numpy(sample_weights).double()

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_y = []
    all_p = []
    all_prob = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        all_y.append(y.cpu().numpy())
        all_p.append(preds.cpu().numpy())
        all_prob.append(probs.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    y_prob = np.concatenate(all_prob)

    return y_true, y_pred, y_prob


def plot_confusion(cm, class_names, out_path, normalize=False):
    cm_to_plot = cm.astype(np.float32).copy()

    if normalize:
        row_sums = cm_to_plot.sum(axis=1, keepdims=True)
        cm_to_plot = cm_to_plot / np.maximum(row_sums, 1.0)

    fig = plt.figure(figsize=(7, 6))
    plt.imshow(cm_to_plot, interpolation="nearest")
    plt.title("Confusion Matrix" + (" (Normalized)" if normalize else ""))
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=30, ha="right")
    plt.yticks(tick_marks, class_names)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_to_plot[i, j]
            text = f"{val:.2f}" if normalize else str(int(val))
            plt.text(j, i, text, ha="center", va="center", fontsize=10)

    plt.tight_layout()
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def train_one_epoch(model, loader, optimizer, criterion, scaler, device, use_amp):
    model.train()
    running = 0.0
    n = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = x.size(0)
        running += loss.item() * bs
        n += bs

    return running / max(1, n)


def train_role_classifier(config_path, local_data_root="/content/hiod_role_crops"):
    cfg = load_config(config_path)
    set_seed(cfg["project"]["seed"])

    device = torch.device(cfg["project"]["device"] if torch.cuda.is_available() else "cpu")
    out_dir = cfg["project"]["out_dir"]
    os.makedirs(out_dir, exist_ok=True)

    data_root = local_data_root if os.path.exists(local_data_root) else cfg["preprocess"]["output_root"]
    train_dir = os.path.join(data_root, "train")
    val_dir = os.path.join(data_root, "val")

    train_tf, val_tf = build_transforms(cfg)

    train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
    val_ds = datasets.ImageFolder(val_dir, transform=val_tf)

    actual = sorted(train_ds.class_to_idx.keys())
    expected = sorted(ROLE_ORDER)
    if actual != expected:
        raise RuntimeError(f"Class folders mismatch. Expected: {expected}, Found: {actual}")

    print("Class mapping:", train_ds.class_to_idx)

    sampler = None
    shuffle = True

    if bool(cfg["train"]["use_weighted_sampler"]):
        sampler = make_weighted_sampler(train_ds)
        shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["train"]["batch_size"]),
        shuffle=shuffle,
        sampler=sampler,
        num_workers=int(cfg["project"]["num_workers"]),
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=int(cfg["train"]["batch_size"]),
        shuffle=False,
        num_workers=int(cfg["project"]["num_workers"]),
        pin_memory=True
    )

    model = make_model(cfg).to(device)

    class_weights = None
    class_counts = None
    if bool(cfg["train"]["use_class_weights"]):
        class_weights, class_counts = compute_class_weights_from_dataset(train_ds, ROLE_ORDER, device)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=float(cfg["train"]["label_smoothing"])
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=float(cfg["train"]["lr"]),
        weight_decay=float(cfg["train"]["weight_decay"])
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=int(cfg["train"]["epochs"]),
        eta_min=float(cfg["train"]["min_lr"])
    )

    use_amp = bool(cfg["train"]["amp"]) and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_f1 = -1.0
    best_epoch = -1
    patience = int(cfg["train"]["early_stopping_patience"])
    no_improve_epochs = 0

    best_path = os.path.join(out_dir, "best.pt")
    last_path = os.path.join(out_dir, "last.pt")
    history_path = os.path.join(out_dir, "history.json")

    with open(os.path.join(out_dir, "config_used.yaml"), "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    history = []

    for epoch in range(1, int(cfg["train"]["epochs"]) + 1):
        t0 = time.time()

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            use_amp=use_amp
        )

        y_true, y_pred, y_prob = evaluate(model, val_loader, device)

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        report = classification_report(
            y_true,
            y_pred,
            target_names=val_ds.classes,
            output_dict=True,
            zero_division=0
        )

        current_lr = optimizer.param_groups[0]["lr"]

        log = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_acc": float(acc),
            "val_macro_f1": float(macro_f1),
            "val_weighted_f1": float(weighted_f1),
            "lr": float(current_lr),
            "time_sec": float(time.time() - t0)
        }
        history.append(log)
        print(json.dumps(log))

        torch.save({
            "model": model.state_dict(),
            "class_to_idx": train_ds.class_to_idx,
            "idx_to_class": {v: k for k, v in train_ds.class_to_idx.items()},
            "epoch": epoch,
            "cfg": cfg,
        }, last_path)

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_epoch = epoch
            no_improve_epochs = 0

            torch.save({
                "model": model.state_dict(),
                "class_to_idx": train_ds.class_to_idx,
                "idx_to_class": {v: k for k, v in train_ds.class_to_idx.items()},
                "epoch": epoch,
                "cfg": cfg,
            }, best_path)

            print(f">>> Saved BEST: {best_path} (macro_f1={best_f1:.4f})")
        else:
            no_improve_epochs += 1

        scheduler.step()

        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2)

        if no_improve_epochs >= patience:
            print(f"Early stopping at epoch {epoch} (best epoch: {best_epoch})")
            break

    # Load best for final report
    best_ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(best_ckpt["model"])

    y_true, y_pred, y_prob = evaluate(model, val_loader, device)

    print("\n=== Final validation report (BEST checkpoint) ===")
    print(classification_report(y_true, y_pred, target_names=val_ds.classes, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    cm_path = os.path.join(out_dir, cfg["eval"]["confusion_matrix_path"])
    ncm_path = os.path.join(out_dir, cfg["eval"]["normalized_confusion_matrix_path"])

    if bool(cfg["eval"]["save_confusion_matrix"]):
        plot_confusion(cm, val_ds.classes, cm_path, normalize=False)
        plot_confusion(cm, val_ds.classes, ncm_path, normalize=True)
        print("Saved confusion matrices:")
        print(" ", cm_path)
        print(" ", ncm_path)

    print("Best checkpoint:", best_path)
    print("Last checkpoint:", last_path)
    print("Best epoch:", best_epoch)
    print("Best macro F1:", best_f1)

    return best_path, last_path

In [ ]:
best_ckpt, last_ckpt = train_role_classifier(CONFIG_PATH)

Class mapping: {'medical_staff': 0, 'other': 1, 'patient': 2}
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 167MB/s]


Class counts: {'medical_staff': 1326, 'other': 391, 'patient': 425}
Class weights: {'medical_staff': 0.39939799904823303, 'other': 1.354480266571045, 'patient': 1.2461217641830444}


/tmp/ipykernel_283/1212606130.py:279: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 1, "train_loss": 0.5595144174513029, "val_acc": 0.7277777777777777, "val_macro_f1": 0.6949868139523313, "val_weighted_f1": 0.7393187346922979, "lr": 0.0003, "time_sec": 48.21196007728577}
>>> Saved BEST: /content/drive/MyDrive/GP Files/HIOD/runs/Run_2/best.pt (macro_f1=0.6950)


...

{"epoch": 31, "train_loss": 0.11495999414491163, "val_acc": 0.8074074074074075, "val_macro_f1": 0.7683832853457623, "val_weighted_f1": 0.8066488254312516, "lr": 8.250000000000003e-05, "time_sec": 24.682013273239136}


/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 32, "train_loss": 0.11407671253136895, "val_acc": 0.7962962962962963, "val_macro_f1": 0.7667339605758244, "val_weighted_f1": 0.799176865854697, "lr": 7.391702899674173e-05, "time_sec": 22.238019227981567}


/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 33, "train_loss": 0.11475627019959672, "val_acc": 0.7925925925925926, "val_macro_f1": 0.756669598939666, "val_weighted_f1": 0.7944709560480941, "lr": 6.572908607777955e-05, "time_sec": 24.219502449035645}
Early stopping at epoch 33 (best epoch: 25)

=== Final validation report (BEST checkpoint) ===
               precision    recall  f1-score   support

medical_staff       0.89      0.81      0.85       340
        other       0.58      0.67      0.62        97
      patient       0.81      0.92      0.86       103

     accuracy                           0.81       540
    macro avg       0.76      0.80      0.78       540
 weighted avg       0.82      0.81      0.81       540

Saved confusion matrices:
  /content/drive/MyDrive/GP Files/HIOD/runs/Run_2/confusion_matrix.png
  /content/drive/MyDrive/GP Files/HIOD/runs/Run_2/confusion_matrix_normalized.png
Best checkpoint: /content/drive/MyDrive/GP Files/HIOD/runs/Run_2/best.pt
Last checkpoint: /content/drive/MyDrive/GP Files/H

In [ ]:
import yaml

config = {
    "project": {
        "seed": 42,
        "device": "cuda",
        "num_workers": 2,
        "out_dir": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_3",
    },

    "data": {
        "base_dir": "/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main",
        "image_label_pairs": [
            {"images": "image/image1", "labels": "labels/label1"},
            {"images": "image/image2", "labels": "labels/label2"},
            {"images": "image/image3", "labels": "labels/label3"},
            {"images": "image/image4", "labels": "labels/label4"},
            {"images": "image/image5", "labels": "labels/label5"},
        ],
        "val_ratio": 0.2,
        "class_map": {
            "staff": "medical_staff",
            "patient": "patient",
            "visitor": "other",
            "person": "other",
        },
    },

    "preprocess": {
        "output_root": "/content/drive/MyDrive/GP Files/HIOD/hiod_role_crops",
        "crop_padding": 0.15,
        "min_box_area": 2500,
        "max_aspect_ratio": 6.0,
        "save_debug_samples": 50,
        "enable_dedup": True,
        "dedup_phash_thresh": 6,
    },

    "train": {
        "model": "mobilenet_v3_large",
        "pretrained": True,
        "num_classes": 3,
        "img_size": 224,
        "batch_size": 16,
        "epochs": 45,
        "lr": 0.0003,
        "min_lr": 0.00001,
        "weight_decay": 0.0001,
        "label_smoothing": 0.03,
        "amp": True,
        "use_class_weights": True,
        "use_weighted_sampler": True,
        "early_stopping_patience": 10,
    },

    "augment": {
        "hflip": 0.5,
        "color_jitter": 0.25,
        "random_erasing": 0.15,
        "random_crop_scale": [0.8, 1.0],
        "rotation_deg": 10,
        "translate": 0.05,
    },

    "eval": {
        "save_confusion_matrix": True,
        "confusion_matrix_path": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_3/confusion_matrix.png",
        "normalized_confusion_matrix_path": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_3/confusion_matrix_normalized.png",
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Wrote config to:", CONFIG_PATH)

Wrote config to: /content/drive/MyDrive/GP Files/HIOD/Code/config.yaml


In [ ]:
best_ckpt, last_ckpt = train_role_classifier(CONFIG_PATH)

Class mapping: {'medical_staff': 0, 'other': 1, 'patient': 2}
Class counts: {'medical_staff': 1326, 'other': 391, 'patient': 425}
Class weights: {'medical_staff': 0.39939799904823303, 'other': 1.354480266571045, 'patient': 1.2461217641830444}


/tmp/ipykernel_283/1212606130.py:279: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 1, "train_loss": 0.5566678878083394, "val_acc": 0.6092592592592593, "val_macro_f1": 0.6028181579020854, "val_weighted_f1": 0.6186281282450865, "lr": 0.0003, "time_sec": 45.46562623977661}
>>> Saved BEST: /content/drive/MyDrive/GP Files/HIOD/runs/Run_3/best.pt (macro_f1=0.6028)


/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 2, "train_loss": 0.41747231663219514, "val_acc": 0.6444444444444445, "val_macro_f1": 0.6377563759586232, "val_weighted_f1": 0.6533017260170818, "lr": 0.0002996467872876745, "time_sec": 25.5539493560791}
>>> Saved BEST: /content/drive/MyDrive/GP Files/HIOD/runs/Run_3/best.pt (macro_f1=0.6378)


...

{"epoch": 27, "train_loss": 0.11530063405674712, "val_acc": 0.7962962962962963, "val_macro_f1": 0.7545313725287318, "val_weighted_f1": 0.796649316594748, "lr": 0.00011992132513804816, "time_sec": 25.86407494544983}


/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 28, "train_loss": 0.11980795578545883, "val_acc": 0.8, "val_macro_f1": 0.754227843547695, "val_weighted_f1": 0.7988236681690125, "lr": 0.00011019253581563264, "time_sec": 24.042441844940186}


/tmp/ipykernel_283/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 29, "train_loss": 0.11340565222596365, "val_acc": 0.8092592592592592, "val_macro_f1": 0.7743327739185237, "val_weighted_f1": 0.8136342804045097, "lr": 0.00010068204395469276, "time_sec": 26.157044887542725}
Early stopping at epoch 29 (best epoch: 19)

=== Final validation report (BEST checkpoint) ===
               precision    recall  f1-score   support

medical_staff       0.89      0.81      0.85       340
        other       0.59      0.70      0.64        97
      patient       0.81      0.89      0.85       103

     accuracy                           0.81       540
    macro avg       0.76      0.80      0.78       540
 weighted avg       0.82      0.81      0.81       540

Saved confusion matrices:
  /content/drive/MyDrive/GP Files/HIOD/runs/Run_3/confusion_matrix.png
  /content/drive/MyDrive/GP Files/HIOD/runs/Run_3/confusion_matrix_normalized.png
Best checkpoint: /content/drive/MyDrive/GP Files/HIOD/runs/Run_3/best.pt
Last checkpoint: /content/drive/MyDrive/GP Files

In [ ]:
import yaml

config = {
    "project": {
        "seed": 42,
        "device": "cuda",
        "num_workers": 2,
        "out_dir": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_4",
    },

    "data": {
        "base_dir": "/content/drive/MyDrive/GP Files/HIOD/Hospital_Scene_Data-main",
        "image_label_pairs": [
            {"images": "image/image1", "labels": "labels/label1"},
            {"images": "image/image2", "labels": "labels/label2"},
            {"images": "image/image3", "labels": "labels/label3"},
            {"images": "image/image4", "labels": "labels/label4"},
            {"images": "image/image5", "labels": "labels/label5"},
        ],
        "val_ratio": 0.2,
        "class_map": {
            "staff": "medical_staff",
            "patient": "patient",
            "visitor": "other",
            "person": "other",
        },
    },

    "preprocess": {
        "output_root": "/content/drive/MyDrive/GP Files/HIOD/hiod_role_crops",
        "crop_padding": 0.15,
        "min_box_area": 2500,
        "max_aspect_ratio": 6.0,
        "save_debug_samples": 50,
        "enable_dedup": True,
        "dedup_phash_thresh": 6,
    },

    "train": {
        "model": "mobilenet_v3_large",
        "pretrained": True,
        "num_classes": 3,
        "img_size": 224,
        "batch_size": 64,
        "epochs": 50,
        "lr": 0.0003,
        "min_lr": 0.00001,
        "weight_decay": 0.0001,
        "label_smoothing": 0.03,
        "amp": True,
        "use_class_weights": True,
        "use_weighted_sampler": True,
        "early_stopping_patience": 10,
    },

    "augment": {
        "hflip": 0.5,
        "color_jitter": 0.25,
        "random_erasing": 0.15,
        "random_crop_scale": [0.8, 1.0],
        "rotation_deg": 10,
        "translate": 0.05,
    },

    "eval": {
        "save_confusion_matrix": True,
        "confusion_matrix_path": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_4/confusion_matrix.png",
        "normalized_confusion_matrix_path": "/content/drive/MyDrive/GP Files/HIOD/runs/Run_4/confusion_matrix_normalized.png",
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Wrote config to:", CONFIG_PATH)

Wrote config to: /content/drive/MyDrive/GP Files/HIOD/Code/config.yaml


In [ ]:
best_ckpt, last_ckpt = train_role_classifier(CONFIG_PATH)

Class mapping: {'medical_staff': 0, 'other': 1, 'patient': 2}
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 140MB/s]


Class counts: {'medical_staff': 1326, 'other': 391, 'patient': 425}
Class weights: {'medical_staff': 0.39939799904823303, 'other': 1.354480266571045, 'patient': 1.2461217641830444}


/tmp/ipykernel_516/1212606130.py:279: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_516/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 1, "train_loss": 0.6109699773855236, "val_acc": 0.7037037037037037, "val_macro_f1": 0.679448981649526, "val_weighted_f1": 0.7140358159127064, "lr": 0.0003, "time_sec": 43.530757665634155}
>>> Saved BEST: /content/drive/MyDrive/GP Files/HIOD/runs/Run_4/best.pt (macro_f1=0.6794)


...

{"epoch": 14, "train_loss": 0.13534329336286147, "val_acc": 0.7481481481481481, "val_macro_f1": 0.7164850205950829, "val_weighted_f1": 0.7540388890310219, "lr": 0.00025425933035965983, "time_sec": 22.382323265075684}


/tmp/ipykernel_516/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 15, "train_loss": 0.1239054479769298, "val_acc": 0.7592592592592593, "val_macro_f1": 0.7311298496474427, "val_weighted_f1": 0.7657109929396848, "lr": 0.00024742647851355997, "time_sec": 21.222205638885498}


/tmp/ipykernel_516/1212606130.py:191: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{"epoch": 16, "train_loss": 0.12626755782981325, "val_acc": 0.7703703703703704, "val_macro_f1": 0.7455558906245252, "val_weighted_f1": 0.7770155265570419, "lr": 0.0002402288615824086, "time_sec": 22.9487566947937}
Early stopping at epoch 16 (best epoch: 6)

=== Final validation report (BEST checkpoint) ===
               precision    recall  f1-score   support

medical_staff       0.86      0.81      0.84       340
        other       0.56      0.62      0.59        97
      patient       0.79      0.86      0.83       103

     accuracy                           0.79       540
    macro avg       0.74      0.76      0.75       540
 weighted avg       0.79      0.79      0.79       540

Saved confusion matrices:
  /content/drive/MyDrive/GP Files/HIOD/runs/Run_4/confusion_matrix.png
  /content/drive/MyDrive/GP Files/HIOD/runs/Run_4/confusion_matrix_normalized.png
Best checkpoint: /content/drive/MyDrive/GP Files/HIOD/runs/Run_4/best.pt
Last checkpoint: /content/drive/MyDrive/GP Files/HIO